# Vision Transformer v1.5 — find the people in a photo, encoder and decoder
A Vision Transformer that labels every pixel of a photo as **person** or **background**, so the people can be painted
in any colour — now shaped like the text Seq2Seq Transformer: an **encoder** reads the photo as patches, and a
**decoder** writes the answer. A convolutional stem turns the photo into 864 patch vectors (24 x 36, one per 8 x 8
pixels), with windows that overlap; the encoder's patches look at each other through attention; then each of 3,456
small cells (4 x 4 pixels) asks the patches what it is — cross-attention, as the text decoder asks the encoder — and
gets two scores, background and person. Positions enter every attention as rotations in two directions (2-D RoPE).

Three changes since v1, each marked `[NEW v1.5]` in the code: the convolutional stem (cell 5), 2-D rotary positions
instead of learned ones (cell 5), and the decoder (cells 5, 6, 10). Everything else — the data, the labelling tool,
the training loop, the scores — is v1's, line for line.

It learns from a dataset of photos whose people are traced by hand — two group photos to start with, 14 people.
Cell 3 is a labelling tool: add your own photos, trace their people, save, and the next run trains on them too.
Two photos are enough to learn these photos and to show every step, not enough to work on any photo: section 8
measures the difference.

## 1. Imports and config

In [ ]:
import base64
import io
import json
import math
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
from PIL import Image, ImageDraw, ImageOps
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb

torch.manual_seed(0)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")                    # the GPU of an Apple-silicon Mac
else:
    device = torch.device("cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

# ---------------- config (all knobs in one place) ----------------
DATA_DIR    = "vision_people_data"                  # the dataset: the photos and outlines.json, the traced people
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:                                        # Colab forgets its disk: keep the dataset on Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/vision_people_data"
    if not os.path.isdir(DATA_DIR):
        print("Upload the vision_people_data folder to the top of your Google Drive (My Drive), then run this again.")
SPLIT       = "both"    # "both": learn every photo | "photo1": learn photo 1, test on the rest | "right_out": hold out the right of each photo | "labels": train / test as marked in the labels
CUT_X       = {"people_1.jpg": 1330, "people_2.jpg": 1352}   # where "right_out" cuts: the two people right of it are held out

IMG_W, IMG_H = 288, 192                             # the picture the model sees: 3:2, like the photos
PATCH        = 8                                    # 8 x 8 pixels per patch -> a 36 x 24 grid of 864 patches
CLASS_NAMES  = ["background", "person"]
NUM_CLASSES  = len(CLASS_NAMES)
IGNORE       = 255                                  # the label of padding: no loss, no score

BASE_W, BASE_H = 432, 288                           # photos are kept 1.5 x larger, so zooming in stays sharp

# how each training picture is made from a photo (cell 4)
SCALE_RANGE  = (0.5, 1.5)                           # each training picture shows its photo at 0.5x to 1.5x
PASTE_PROB   = 0.5                                  # half the pictures get one extra person pasted in
PASTE_SCALE  = (0.5, 1.4)                           # the pasted person's own zoom
PASTE_AT_X   = (-0.9, 0.9)                          # where their centre lands, across (the picture runs from -1 to 1)
PASTE_AT_Y   = (-0.3, 0.4)                          # and down: around the middle, where people stand
BRIGHTNESS   = (0.75, 1.25)                         # colour changes, drawn anew for every picture
CONTRAST     = (0.75, 1.25)
SATURATION   = (0.7, 1.3)

D_MODEL      = 128                                  # numbers in each patch's vector
NUM_HEADS    = 4                                    # attentions side by side in each layer
NUM_LAYERS   = 4                                    # encoder layers, one after another
D_FF         = 512                                  # width of each layer's small network
DROPOUT      = 0.1                                  # share of numbers zeroed while training

# [NEW v1.5] the encoder–decoder's own knobs (cell 5)
STEM_CH      = (32, 64)                             # the convolutional stem's channels: 3 -> 32 -> 64 -> D_MODEL
CELL         = 4                                    # the decoder labels 4 x 4-pixel cells: a 72 x 48 grid of 3,456
DEC_LAYERS   = 2                                    # decoder layers: each cell asks the patches twice
ROPE_BASE    = 100.0                                # 2-D rotary positions: the slowest pair turns once in ~350 patches

SAMPLES_PER_EPOCH = 1024                            # pictures made per epoch: there are only two photos to go round
BATCH_SIZE   = 32
EPOCHS       = 60
LR           = 1e-3
WEIGHT_DECAY = 0.05
CKPT_PATH    = "people_vit_v15.pt"                  # [NEW v1.5] its own file: v1's people_vit.pt is left alone

## 2. The dataset: every labelled photo
`PeopleDataset` is the one place the labels live. `outlines.json` lists the photos — file, size, train or test, the
order their people are drawn in — and every person as one or more polygons: corner points along their outline, in the
photo's own pixels. A person can also have holes: places inside the outline that are not them, like the wall seen
between an arm and the body. A mask is always made from the polygons: every pixel inside a person's outline, and not in
one of their holes, gets their number (1, 2, …), drawn back to front so the person in front keeps the pixels where two
overlap — and a hole shows whatever is behind it. This model only needs person or not, so later cells turn every
number above 0 into "person"; the numbers are kept for a model that tells people apart. Adding a photo, changing a
person and saving all go through this class — the labelling tool in cell 3 does.

In [ ]:
def image_to_tensor(image):
    """A PIL image as a (3, H, W) tensor of floats in [0, 1]: colour first, as PyTorch expects."""
    return torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255


class PeopleDataset:
    """Every labelled photo in a folder: outlines.json plus the photo files it names."""

    def __init__(self, root):
        self.root = root
        self.path = os.path.join(root, "outlines.json")
        data = json.load(open(self.path)) if os.path.exists(self.path) else {"images": [], "people": []}
        self.about = data.get("about", "People traced as polygons in photo pixels.")
        self.images, self.people = data["images"], data["people"]
        for im in self.images:
            im.setdefault("split", "train")
            im.setdefault("draw_order", [p["person"] for p in self.people if p["image"] == im["id"]])

    def __len__(self):
        return len(self.images)

    def names(self):
        return [im["file"] for im in self.images]

    def index(self, name):
        return self.names().index(name)

    def people_of(self, i):
        """Photo i's people, in the order they are drawn: the ones behind first."""
        mine = {p["person"]: p for p in self.people if p["image"] == self.images[i]["id"]}
        return [mine[k] for k in self.images[i]["draw_order"] if k in mine]

    def photo(self, i, size=None):
        """Photo i as a (3, H, W) tensor in [0, 1], optionally resized to size = (width, height)."""
        image = Image.open(os.path.join(self.root, self.images[i]["file"])).convert("RGB")
        if size is not None:
            image = image.resize(size, Image.LANCZOS)
        return image_to_tensor(image)

    def mask(self, i, size=None):
        """Photo i's mask as an (H, W) tensor, made from its polygons: 0 = background, k = person k. Back to front,
        each person is filled in without their holes, so a hole shows what is behind it."""
        im = self.images[i]
        mask = Image.new("L", (im["width"], im["height"]), 0)
        for person in self.people_of(i):
            layer = Image.new("L", mask.size, 0)
            draw = ImageDraw.Draw(layer)
            for polygon in person["polygons"]:
                draw.polygon([tuple(q) for q in polygon], fill=255)
            for hole in person.get("holes", []):
                draw.polygon([tuple(q) for q in hole], fill=0)
            mask.paste(person["person"], mask=layer)
        if size is not None:
            mask = mask.resize(size, Image.NEAREST)     # nearest: a person's number must never be blended
        return torch.from_numpy(np.array(mask)).long()

    def __getitem__(self, i):
        """Photo i as a (3, H, W) tensor in [0, 1], its mask (H, W), and its entry in outlines.json."""
        return self.photo(i), self.mask(i), self.images[i]

    def add_photo(self, name, data):
        """Saves a new photo (its bytes) into the folder — upright, as it is shown — with no people yet."""
        image = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert("RGB")
        stem, ext = os.path.splitext(os.path.basename(name))
        ext = ".png" if ext.lower() == ".png" else ".jpg"   # PNG stays PNG; everything else is saved as JPEG
        while stem + ext in self.names():                   # never overwrite a photo: "_2" until the name is free
            stem += "_2"
        name = stem + ext
        image.save(os.path.join(self.root, name), quality=95)
        new_id = max([im["id"] for im in self.images], default=0) + 1
        self.images.append({"id": new_id, "file": name, "width": image.width, "height": image.height,
                            "split": "train", "draw_order": []})
        return len(self.images) - 1

    def remove_photo(self, i):
        """Takes photo i and its people out of the dataset (its file stays in the folder)."""
        gone = self.images.pop(i)["id"]
        self.people = [p for p in self.people if p["image"] != gone]

    def set_people(self, i, people):
        """Replaces photo i's people: a list of {"person": k, "polygons": [[[x, y], ...], ...], "holes": [...]},
        back to front ("holes" may be left out)."""
        n = self.images[i]["id"]
        self.people = [p for p in self.people if p["image"] != n]
        for person in people:
            entry = {"image": n, "person": int(person["person"]), "polygons": person["polygons"]}
            if person.get("holes"):
                entry["holes"] = person["holes"]
            entry["vertices"] = sum(len(q) for q in person["polygons"] + person.get("holes", []))
            self.people.append(entry)
        self.images[i]["draw_order"] = [int(person["person"]) for person in people]

    def save(self):
        json.dump({"about": self.about, "images": self.images, "people": self.people}, open(self.path, "w"), indent=1)


def load_photo(name, size=None):
    """A photo of the dataset by its file name (the later cells use names)."""
    return dataset.photo(dataset.index(name), size)


def load_mask(name, size=None):
    return dataset.mask(dataset.index(name), size)


dataset = PeopleDataset(DATA_DIR)
PHOTOS = dataset.names()
first = next(iter(dataset.people_of(0)), None) if len(dataset) else None   # none yet in a fresh dataset
if first:
    print(f"Person {first['person']} of {PHOTOS[0]}: {sum(len(q) for q in first['polygons'])} corner points "
          f"-> {int((dataset.mask(0) == first['person']).sum()):,} pixels")

photos = {name: load_photo(name) for name in PHOTOS}                # full size: what the tests are scored on
masks = {name: load_mask(name) for name in PHOTOS}
for name in PHOTOS:
    m, n = masks[name], len(masks[name].unique()) - 1
    print(f"{name}: {m.shape[1]} x {m.shape[0]} pixels, {n} {'person' if n == 1 else 'people'}, "
          f"{(m > 0).float().mean():.1%} of the pixels are people ({dataset.images[dataset.index(name)]['split']})")

## 3. The labelling tool
Run this cell to open the dataset in a small editor. **Left:** the photos, each with its number of people and its
train / test mark (click the mark to switch it); the open photo lists its people. **Right:** the photo with every
person's outline. Zoom with the mouse wheel or + / − / Fit, and drag the empty canvas to move around.

- **Edit an outline:** click a person: their round points appear, and a small square in the middle of every edge. Drag
  a point to move it. Drag a square to pull a new point out of that edge (a click adds it right there) — that is how
  an outline gets more points exactly where it needs them; zoom in and the shorter edges get squares too. *More
  points* adds one to every edge of the person at once. Select a point and press Delete to remove it. Ctrl/Cmd + Z
  undoes.
- **A new person:** click *New person*, click the corner points along their outline, double-click or press Enter to
  close it (Esc cancels). *Add part* draws one more outline for the selected person — a hand on the far side of
  someone, say.
- **A hole:** where an arm makes a loop — a hand on the hip, a hand at the glasses — the wall seen through it is not
  the person. Select them, click *Add hole* and click around the gap: it is drawn dashed and left out of their mask.
  Its points are edited like any others; deleting a hole's points down to two removes the hole.
- **Who is in front:** the list goes back to front, and where two outlines overlap the one in front keeps the pixels.
  *Move behind* / *Move in front* change the selected person's place.
- **More photos:** *Add photos…* copies them into the dataset folder, with no people yet (a photo longer than 2400
  pixels is shrunk first). Until it has people, training leaves it out.
- **Save** writes `outlines.json`. Run the cells below again and the model trains on what you saved.

The tool needs `anywidget` (`%pip install anywidget`); it works in Jupyter, VS Code and Colab. Outside a notebook this
cell does nothing.

In [ ]:
IN_NOTEBOOK = False
try:
    import anywidget
except ImportError:
    anywidget = None
    if IN_COLAB:                                     # Colab starts without it every session: install it there
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anywidget"])
        import anywidget

if anywidget is None:
    print("The labelling tool needs anywidget: run  %pip install anywidget  and then this cell again.")
else:
    import traitlets
    from IPython import get_ipython
    from IPython.display import display
    IN_NOTEBOOK = get_ipython() is not None
    if IN_NOTEBOOK and IN_COLAB:
        from google.colab import output
        output.enable_custom_widget_manager()       # Colab shows widgets that are not its own only with this

    class LabelTool(anywidget.AnyWidget):
        """The dataset in an editor: a tree of photos and people, and the photo with its outlines on a canvas."""
        _esm = r"""// The labelling tool of the vision notebook (cell 3): an anywidget front end. nbvis.py puts this file, as it is, into
// the notebook cell as LabelTool._esm. Left: the dataset as a tree — photos, then the people of the open photo. Right:
// the photo on a canvas with every person's outline; zoom, pan, move / add / remove points (a square handle in the
// middle of every edge pulls out a new one; More points adds one to every edge), draw new people, new parts of a
// person or holes (a gap inside the outline that is not the person: dashed, left out of the fill), undo. A person's
// rings are their polygons, then their holes: sel.part counts through both. Edits stay here until Save sends them to Python, which writes outlines.json.
// Coordinates are always the photo's own pixels; the canvas only changes how they are shown. A person keeps their
// colour (it follows their number). The list order of people is the drawing order, back to front.
const BIG = 2400;                      // a photo longer than this is shrunk in the browser before it is sent
const COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4', '#42d4f4', '#f032e6', '#9a6324', '#469990',
                '#800000', '#808000', '#000075'];

function colour(pp) { return COLORS[(pp.person - 1) % COLORS.length]; }

function el(tag, attrs, parent, text) {
  const e = document.createElement(tag);
  for (const k in attrs || {}) e.setAttribute(k, attrs[k]);
  if (text !== undefined) e.textContent = text;
  if (parent) parent.appendChild(e);
  return e;
}

function render({ model, el: root }) {
  root.classList.add('lt');
  root.setAttribute('data-lm-suppress-shortcuts', 'true');   // JupyterLab: keys typed here are the tool's
  const side = el('div', { class: 'lt-side' }, root);
  const main = el('div', { class: 'lt-main' }, root);
  const bar = el('div', { class: 'lt-bar' }, main);
  const wrap = el('div', { class: 'lt-wrap' }, main);
  const canvas = el('canvas', { class: 'lt-canvas', tabindex: '0', 'aria-label': 'the photo and its outlines' }, wrap);
  const status = el('div', { class: 'lt-status', 'aria-live': 'polite' }, main);
  const ctx = canvas.getContext('2d');
  const layer = document.createElement('canvas'), lctx = layer.getContext('2d');   // a person with holes is filled here
  const img = new Image();
  let dpr = 1;

  let photos = [], cur = 0, people = [], file = null;
  const edits = {};                    // file -> people, changed and not saved yet
  const undo = {};                     // file -> earlier versions of people (JSON)
  const step = {}, savedStep = {};     // file -> changes made (undos count back), and that count when last saved
  let sel = { p: -1, part: -1, v: -1 };
  let draw = null;                     // drawing: {target: 'new' | person index, hole: true for a hole, pts: [[x, y], ...]}
  let view = { s: 1, ox: 0, oy: 0 }, drag = null, mouse = null, confirmDel = 0, pulled = 0;

  // ------------------------------------------------------------------ the data
  function load() {
    photos = model.get('photos') || [];
    cur = model.get('current') || 0;
    const ph = photos[cur];
    file = ph ? ph.file : null;
    people = file && edits[file] ? edits[file] : JSON.parse(JSON.stringify(model.get('people') || []))
      .map(pp => ({ person: pp.person, polygons: pp.polygons, holes: pp.holes || [] }));
    sel = { p: -1, part: -1, v: -1 };
    draw = null;
  }
  function dirtyCount() { return Object.keys(edits).length; }
  function rings(pp) { return pp.polygons.concat(pp.holes); }          // the same arrays: editing one edits the person
  function isHole(pp, part) { return part >= pp.polygons.length; }
  function snapshot() {
    (undo[file] = undo[file] || []).push(JSON.stringify(people));
    if (undo[file].length > 100) undo[file].shift();
    step[file] = (step[file] || 0) + 1;
  }
  function changed() {
    if ((step[file] || 0) === (savedStep[file] || 0)) delete edits[file];   // undone back to what is saved
    else edits[file] = people;
    tree(); toolbar(); paint();
  }
  function say(t) { status.textContent = t; }

  // ------------------------------------------------------------------ the tree (left)
  function tree() {
    side.innerHTML = '';
    const top = el('div', { class: 'lt-top' }, side);
    const add = el('label', { class: 'lt-btn' }, top, 'Add photos…');
    const input = el('input', { type: 'file', accept: 'image/*', multiple: '' }, add);
    input.addEventListener('change', () => {
      const files = [...input.files];
      say('Adding ' + files.length + ' photo(s)…');
      files.forEach(upload);
      input.value = '';
    });
    const save = el('button', { class: 'lt-btn lt-save', type: 'button' }, top,
                    dirtyCount() ? 'Save (' + dirtyCount() + ')' : 'Saved');
    save.disabled = !dirtyCount();
    save.addEventListener('click', () => {
      model.send({ kind: 'save', edits: edits });
      say('Saving…');
    });
    el('div', { class: 'lt-head' }, side, 'Dataset · ' + photos.length + ' photo' + (photos.length === 1 ? '' : 's'));
    photos.forEach((ph, i) => {
      const row = el('div', { class: 'lt-photo' + (i === cur ? ' on' : '') }, side);
      if (ph.thumb) el('img', { src: ph.thumb, alt: '' }, row);
      const t = el('div', { class: 'lt-ptext' }, row);
      el('div', { class: 'lt-name' }, t, ph.file + (edits[ph.file] ? ' •' : ''));
      const n = edits[ph.file] ? edits[ph.file].length : ph.count;
      el('div', { class: 'lt-sub' }, t, ph.width + ' × ' + ph.height + ' · ' + n + ' people');
      const sp = el('button', { class: 'lt-split ' + ph.split, type: 'button', title: 'train or test: click to switch' },
                    row, ph.split);
      sp.addEventListener('click', e => {
        e.stopPropagation();
        model.send({ kind: 'split', file: ph.file, split: ph.split === 'train' ? 'test' : 'train' });
      });
      row.addEventListener('click', () => { if (i !== cur) { model.set('current', i); model.save_changes(); } });
      if (i === cur) {
        const list = el('div', { class: 'lt-people' }, side);
        if (people.length > 1) el('div', { class: 'lt-sub' }, list, 'People, back to front:');
        people.forEach((pp, k) => {
          const pts = rings(pp).reduce((a, q) => a + q.length, 0);
          const r = el('div', { class: 'lt-person' + (k === sel.p ? ' on' : '') }, list);
          el('i', { style: 'background:' + colour(pp) }, r);
          el('span', {}, r, 'Person ' + pp.person + ' · ' + pts + ' points' +
             (pp.polygons.length > 1 ? ' · ' + pp.polygons.length + ' parts' : '') +
             (pp.holes.length ? ' · ' + pp.holes.length + (pp.holes.length > 1 ? ' holes' : ' hole') : ''));
          r.addEventListener('click', () => { sel = { p: k, part: -1, v: -1 }; tree(); toolbar(); paint(); });
        });
        if (!people.length) el('div', { class: 'lt-sub lt-empty' }, list, 'No people yet: click “New person”.');
      }
    });
  }

  // ------------------------------------------------------------------ the toolbar
  function button(label, title, fn, disabled) {
    const b = el('button', { class: 'lt-btn', type: 'button', title: title }, bar, label);
    b.disabled = !!disabled;
    b.addEventListener('click', () => { fn(); canvas.focus(); });
    return b;
  }
  function toolbar() {
    bar.innerHTML = '';
    button('−', 'zoom out', () => zoomAt(0.8));
    button('+', 'zoom in', () => zoomAt(1.25));
    button('Fit', 'the whole photo', fit);
    el('span', { class: 'lt-sep' }, bar);
    button(draw && draw.target === 'new' ? 'Drawing…' : 'New person', 'click the outline’s points; double-click or Enter to close',
           () => startDraw('new'), !file);
    button('Add part', 'another outline for the selected person (a hand on the far side, say)',
           () => startDraw(sel.p), sel.p < 0);
    button(draw && draw.hole ? 'Drawing hole…' : 'Add hole', 'a gap inside the selected person that is not them — the wall ' +
           'seen between an arm and the body: click around it', () => startDraw(sel.p, true), sel.p < 0);
    button('More points', 'a new point in the middle of every edge of the selected person, to shape the outline more finely',
           morePoints, sel.p < 0);
    button('Delete point', 'or press Delete', deletePoint, sel.v < 0);
    button(confirmDel ? 'Really delete?' : 'Delete person', 'remove the selected person', deletePerson, sel.p < 0);
    button('Move behind', 'one step further back: where two outlines overlap, the one in front keeps the pixels',
           () => move(-1), sel.p <= 0);
    button('Move in front', 'one step further forward', () => move(1), sel.p < 0 || sel.p >= people.length - 1);
    button('Undo', 'Ctrl/Cmd + Z', doUndo, !(undo[file] && undo[file].length));
    el('span', { class: 'lt-sep' }, bar);
    button('Remove photo', 'take this photo out of the dataset (the file stays in the folder)', () => {
      if (!file) return;
      if (confirmDel !== 'photo') { confirmDel = 'photo'; say('Click “Remove photo” again to take ' + file + ' out of the dataset.'); return; }
      confirmDel = 0;
      delete edits[file];
      model.send({ kind: 'remove', file: file });
    }, !file);
  }

  // ------------------------------------------------------------------ the canvas (right)
  function size() {
    const r = wrap.getBoundingClientRect();
    dpr = window.devicePixelRatio || 1;
    const w = Math.max(320, r.width), h = Math.max(320, Math.round(w * 0.62));
    canvas.style.width = w + 'px'; canvas.style.height = h + 'px';
    canvas.width = Math.round(w * dpr); canvas.height = Math.round(h * dpr);
    ctx.setTransform(dpr, 0, 0, dpr, 0, 0);
    return { w: w, h: h };
  }
  function fit() {
    const ph = photos[cur];
    if (!ph) return;
    const c = size();
    view.s = Math.min(c.w / ph.width, c.h / ph.height) * 0.98;
    view.ox = (c.w - ph.width * view.s) / 2;
    view.oy = (c.h - ph.height * view.s) / 2;
    paint();
  }
  function toScreen(p) { return [p[0] * view.s + view.ox, p[1] * view.s + view.oy]; }
  function toPhoto(x, y) { return [(x - view.ox) / view.s, (y - view.oy) / view.s]; }
  function zoomAt(f, x, y) {
    const c = { w: canvas.clientWidth, h: canvas.clientHeight };
    if (x === undefined) { x = c.w / 2; y = c.h / 2; }
    const p = toPhoto(x, y);
    view.s = Math.max(0.02, Math.min(40, view.s * f));
    view.ox = x - p[0] * view.s; view.oy = y - p[1] * view.s;
    paint();
  }
  function poly(pts) {
    ctx.beginPath();
    pts.forEach((q, i) => { const s = toScreen(q); if (i) ctx.lineTo(s[0], s[1]); else ctx.moveTo(s[0], s[1]); });
  }
  function ring(c, pts) {                                     // a closed outline on context c, in screen pixels
    c.beginPath();
    pts.forEach((q, i) => { const s = toScreen(q); if (i) c.lineTo(s[0], s[1]); else c.moveTo(s[0], s[1]); });
    c.closePath();
  }
  function paint() {
    const W = canvas.clientWidth, H = canvas.clientHeight;
    ctx.clearRect(0, 0, W, H);
    ctx.fillStyle = '#20242c'; ctx.fillRect(0, 0, W, H);
    const ph = photos[cur];
    if (!ph) { ctx.fillStyle = '#cfd5df'; ctx.font = '14px sans-serif'; ctx.fillText('No photos yet: click “Add photos…”.', 20, 30); return; }
    if (img.complete && img.naturalWidth) ctx.drawImage(img, view.ox, view.oy, ph.width * view.s, ph.height * view.s);
    people.forEach((pp, k) => {
      const col = colour(pp), on = k === sel.p;
      if (pp.holes.length) {                                   // filled on the layer, holes erased, then laid over
        if (layer.width !== canvas.width || layer.height !== canvas.height) { layer.width = canvas.width; layer.height = canvas.height; }
        lctx.setTransform(1, 0, 0, 1, 0, 0); lctx.clearRect(0, 0, layer.width, layer.height);
        lctx.setTransform(dpr, 0, 0, dpr, 0, 0); lctx.fillStyle = col;
        lctx.globalCompositeOperation = 'source-over';
        pp.polygons.forEach(pts => { ring(lctx, pts); lctx.fill(); });
        lctx.globalCompositeOperation = 'destination-out';
        pp.holes.forEach(pts => { ring(lctx, pts); lctx.fill(); });
        lctx.globalCompositeOperation = 'source-over';
        ctx.save(); ctx.setTransform(1, 0, 0, 1, 0, 0); ctx.globalAlpha = on ? 0.34 : 0.2; ctx.drawImage(layer, 0, 0); ctx.restore();
      } else {
        pp.polygons.forEach(pts => { ring(ctx, pts); ctx.globalAlpha = on ? 0.34 : 0.2; ctx.fillStyle = col; ctx.fill(); });
      }
      ctx.globalAlpha = 1; ctx.lineWidth = on ? 2.5 : 1.6; ctx.strokeStyle = col;
      pp.polygons.forEach(pts => { ring(ctx, pts); ctx.stroke(); });
      ctx.setLineDash([5, 4]);
      pp.holes.forEach(pts => { ring(ctx, pts); ctx.stroke(); });
      ctx.setLineDash([]);
    });
    if (sel.p >= 0 && people[sel.p]) {
      mids().forEach(m => {                                      // a square in the middle of every edge: drag it to add a point
        ctx.globalAlpha = 0.9; ctx.fillStyle = '#ffffff'; ctx.fillRect(m.x - 3.5, m.y - 3.5, 7, 7);
        ctx.globalAlpha = 1; ctx.lineWidth = 1.5; ctx.strokeStyle = colour(people[sel.p]); ctx.strokeRect(m.x - 3.5, m.y - 3.5, 7, 7);
      });
      rings(people[sel.p]).forEach((pts, part) => pts.forEach((q, v) => {
        const s = toScreen(q), hot = part === sel.part && v === sel.v;
        ctx.beginPath(); ctx.arc(s[0], s[1], hot ? 6 : 4, 0, 7);
        ctx.fillStyle = hot ? '#ffffff' : colour(people[sel.p]); ctx.fill();
        ctx.lineWidth = 1.5; ctx.strokeStyle = '#000000'; ctx.stroke();
      }));
    }
    if (draw) {
      const pts = mouse ? draw.pts.concat([toPhoto(mouse[0], mouse[1])]) : draw.pts;
      if (pts.length) { poly(pts); ctx.lineWidth = 2; ctx.strokeStyle = '#ffffff'; ctx.setLineDash([6, 4]); ctx.stroke(); ctx.setLineDash([]); }
      draw.pts.forEach(q => { const s = toScreen(q); ctx.beginPath(); ctx.arc(s[0], s[1], 3.5, 0, 7); ctx.fillStyle = '#ffffff'; ctx.fill(); });
    }
    const n = people.length, pts = people.reduce((a, pp) => a + rings(pp).reduce((b, q) => b + q.length, 0), 0);
    if (!drag && !draw) say(ph.file + ' · ' + n + ' people · ' + pts + ' points · zoom ' + (view.s * 100).toFixed(0) + '%' +
                             (dirtyCount() ? ' · unsaved changes in ' + dirtyCount() + ' photo(s)' : '') +
                             (sel.p >= 0 ? ' · drag a round point to move it, a square to add one' : ''));
  }

  // ------------------------------------------------------------------ editing
  function hitVertex(x, y) {
    if (sel.p < 0 || !people[sel.p]) return null;
    let best = null;
    rings(people[sel.p]).forEach((pts, part) => pts.forEach((q, v) => {
      const s = toScreen(q), d = Math.hypot(s[0] - x, s[1] - y);
      if (d < 8 && (!best || d < best.d)) best = { part: part, v: v, d: d };
    }));
    return best;
  }
  function mids() {
    // the middle of every edge of the selected person, on screen; an edge shorter than 22 screen pixels has none
    // (zoom in to get them) — so the handles never crowd the points
    const out = [];
    if (sel.p < 0 || !people[sel.p]) return out;
    rings(people[sel.p]).forEach((pts, part) => pts.forEach((q, v) => {
      const a = toScreen(q), b = toScreen(pts[(v + 1) % pts.length]);
      if (Math.hypot(b[0] - a[0], b[1] - a[1]) >= 22) out.push({ part: part, v: v, x: (a[0] + b[0]) / 2, y: (a[1] + b[1]) / 2 });
    }));
    return out;
  }
  function hitMid(x, y) {
    let best = null;
    mids().forEach(m => { const d = Math.hypot(m.x - x, m.y - y); if (d < 7 && (!best || d < best.d)) best = Object.assign(m, { d: d }); });
    return best;
  }
  function hitEdge(x, y) {
    if (sel.p < 0 || !people[sel.p]) return null;
    let best = null;
    rings(people[sel.p]).forEach((pts, part) => pts.forEach((q, v) => {
      const a = toScreen(q), b = toScreen(pts[(v + 1) % pts.length]);
      const dx = b[0] - a[0], dy = b[1] - a[1], L = dx * dx + dy * dy || 1;
      const t = Math.max(0, Math.min(1, ((x - a[0]) * dx + (y - a[1]) * dy) / L));
      const d = Math.hypot(a[0] + t * dx - x, a[1] + t * dy - y);
      if (d < 7 && (!best || d < best.d)) best = { part: part, v: v, d: d };
    }));
    return best;
  }
  function inside(pts, p) {
    let c = false;
    for (let i = 0, j = pts.length - 1; i < pts.length; j = i++) {
      const a = pts[i], b = pts[j];
      if ((a[1] > p[1]) !== (b[1] > p[1]) && p[0] < (b[0] - a[0]) * (p[1] - a[1]) / (b[1] - a[1]) + a[0]) c = !c;
    }
    return c;
  }
  function hitPerson(x, y) {
    const p = toPhoto(x, y);
    for (let k = people.length - 1; k >= 0; k--)
      if (people[k].polygons.some(pts => inside(pts, p)) && !people[k].holes.some(pts => inside(pts, p))) return k;
    return -1;
  }
  function round(p) { return [Math.round(p[0]), Math.round(p[1])]; }
  function startDraw(target, hole) {
    if (target !== 'new' && target < 0) return;
    draw = { target: target, hole: !!hole, pts: [] };
    toolbar(); paint();
    say(hole ? 'Click around the gap. Double-click or Enter closes the hole; Esc cancels.'
             : 'Click the outline’s points. Double-click or Enter closes it; Esc cancels.');
  }
  function finishDraw() {
    if (!draw) return;
    if (draw.pts.length < 3) { say('An outline needs at least 3 points.'); return; }
    snapshot();
    const pts = draw.pts.map(round);
    if (draw.target === 'new') {
      const id = people.reduce((m, pp) => Math.max(m, pp.person), 0) + 1;
      people.push({ person: id, polygons: [pts], holes: [] });
      sel = { p: people.length - 1, part: 0, v: -1 };
    } else if (draw.hole) {
      const pp = people[draw.target];
      pp.holes.push(pts);
      sel = { p: draw.target, part: pp.polygons.length + pp.holes.length - 1, v: -1 };
    } else {
      const pp = people[draw.target];
      pp.polygons.push(pts);
      sel = { p: draw.target, part: pp.polygons.length - 1, v: -1 };
    }
    draw = null;
    changed();
  }
  function morePoints() {
    // a new point in the middle of every edge of the selected person (edges under 4 photo pixels are left alone)
    if (sel.p < 0) return;
    snapshot();
    const pp = people[sel.p];
    const finer = pts => pts.flatMap((q, v) => {
      const r = pts[(v + 1) % pts.length];
      return Math.hypot(r[0] - q[0], r[1] - q[1]) >= 4 ? [q, round([(q[0] + r[0]) / 2, (q[1] + r[1]) / 2])] : [q];
    });
    pp.polygons = pp.polygons.map(finer);
    pp.holes = pp.holes.map(finer);
    sel.v = -1;
    changed();
  }
  function deletePoint() {
    if (sel.p < 0 || sel.v < 0) return;
    const pp = people[sel.p], pts = rings(pp)[sel.part];
    if (pts.length <= 3) {
      if (!isHole(pp, sel.part) && pp.polygons.length === 1) { say('An outline keeps at least 3 points: delete the person instead.'); return; }
      snapshot();                                              // a hole, or a second part, with too few points left: gone
      if (isHole(pp, sel.part)) pp.holes.splice(sel.part - pp.polygons.length, 1); else pp.polygons.splice(sel.part, 1);
      sel.part = -1; sel.v = -1;
      changed();
      return;
    }
    snapshot();
    pts.splice(sel.v, 1);
    sel.v = -1;
    changed();
  }
  function deletePerson() {
    if (sel.p < 0) return;
    if (confirmDel !== 'person') { confirmDel = 'person'; toolbar(); say('Click “Really delete?” to remove person ' + people[sel.p].person + '.'); return; }
    confirmDel = 0;
    snapshot();
    people.splice(sel.p, 1);
    sel = { p: -1, part: -1, v: -1 };
    changed();
  }
  function move(step) {
    const k = sel.p, j = sel.p + step;
    if (k < 0 || j < 0 || j >= people.length) return;
    snapshot();
    const t = people[k]; people[k] = people[j]; people[j] = t;
    sel = { p: j, part: -1, v: -1 };
    changed();
  }
  async function upload(f) {
    // sent as it is, unless longer than BIG pixels: then shrunk here first, upright as shown — a phone photo would be
    // a message of many megabytes, and the model trains on 432 x 288 pictures anyway
    let data = null;
    try {
      const bm = await createImageBitmap(f);
      const k = BIG / Math.max(bm.width, bm.height);
      if (k < 1) {
        const c = document.createElement('canvas');
        c.width = Math.round(bm.width * k); c.height = Math.round(bm.height * k);
        c.getContext('2d').drawImage(bm, 0, 0, c.width, c.height);
        data = c.toDataURL('image/jpeg', 0.92);
      }
      bm.close();
    } catch (err) { say('Could not read ' + f.name + ' as a picture.'); return; }
    if (!data) data = await new Promise(ok => { const r = new FileReader(); r.onload = () => ok(String(r.result)); r.readAsDataURL(f); });
    model.send({ kind: 'upload', name: f.name, data: data.split(',')[1] });
  }
  function doUndo() {
    const st = undo[file];
    if (!st || !st.length) return;
    people = JSON.parse(st.pop());
    step[file] = (step[file] || 0) - 1;
    sel = { p: sel.p < people.length ? sel.p : -1, part: -1, v: -1 };     // the same person stays selected
    changed();
  }

  function xy(e) { const r = canvas.getBoundingClientRect(); return [e.clientX - r.left, e.clientY - r.top]; }
  canvas.addEventListener('pointerdown', e => {
    canvas.focus();
    const p = xy(e);
    if (confirmDel) { confirmDel = 0; toolbar(); }
    if (draw) return;                                          // points are added on click
    const hv = hitVertex(p[0], p[1]);
    const hm = hv ? null : hitMid(p[0], p[1]);
    if (hv) {
      sel.part = hv.part; sel.v = hv.v;
      drag = { kind: 'vertex', moved: false };
    } else if (hm) {                                           // a new point in the middle of that edge, already held
      snapshot();
      const pts = rings(people[sel.p])[hm.part], a = pts[hm.v], b = pts[(hm.v + 1) % pts.length];
      pts.splice(hm.v + 1, 0, round([(a[0] + b[0]) / 2, (a[1] + b[1]) / 2]));
      sel.part = hm.part; sel.v = hm.v + 1;
      edits[file] = people;
      drag = { kind: 'vertex', moved: true };                 // the snapshot holds the outline before the new point
      pulled = Date.now();
    } else {
      const k = hitPerson(p[0], p[1]);
      if (k >= 0 && k !== sel.p) { sel = { p: k, part: -1, v: -1 }; tree(); }
      else if (k < 0) sel.v = -1;
      drag = { kind: 'pan', x: p[0], y: p[1], ox: view.ox, oy: view.oy };
    }
    canvas.setPointerCapture(e.pointerId);
    toolbar(); paint();
  });
  canvas.addEventListener('pointermove', e => {
    const p = xy(e);
    mouse = p;
    if (drag && drag.kind === 'vertex') {
      if (!drag.moved) { snapshot(); drag.moved = true; }
      rings(people[sel.p])[sel.part][sel.v] = round(toPhoto(p[0], p[1]));
      edits[file] = people;
      paint();
    } else if (drag && drag.kind === 'pan') {
      view.ox = drag.ox + p[0] - drag.x; view.oy = drag.oy + p[1] - drag.y;
      paint();
    } else if (draw) paint();
    canvas.style.cursor = drag && drag.kind === 'vertex' ? 'grabbing' : draw || drag ? 'crosshair' :
      hitVertex(p[0], p[1]) ? 'grab' : hitMid(p[0], p[1]) ? 'copy' : 'crosshair';
  });
  canvas.addEventListener('pointerup', () => {
    if (drag && drag.kind === 'vertex' && drag.moved) { drag = null; changed(); return; }
    drag = null;
    paint();
  });
  canvas.addEventListener('click', e => {
    if (!draw) return;
    const p = xy(e);
    if (e.detail > 1) return;                                  // the second click of a double-click closes instead
    draw.pts.push(toPhoto(p[0], p[1]));
    paint();
  });
  canvas.addEventListener('dblclick', e => {
    const p = xy(e);
    if (draw) { finishDraw(); return; }
    if (Date.now() - pulled < 600) return;                     // a double-click on a square: its first click added the point
    const he = hitEdge(p[0], p[1]);
    if (!he) return;
    snapshot();
    rings(people[sel.p])[he.part].splice(he.v + 1, 0, round(toPhoto(p[0], p[1])));
    sel.part = he.part; sel.v = he.v + 1;
    changed();
  });
  canvas.addEventListener('wheel', e => {
    e.preventDefault();
    const p = xy(e);
    // a mouse notch is about 100 px of delta, a trackpad sends many small ones: zoom by how far the wheel went
    const px = e.deltaY * (e.deltaMode === 1 ? 16 : e.deltaMode === 2 ? 400 : 1);
    zoomAt(Math.min(2, Math.max(0.5, Math.exp(-px * 0.0014))), p[0], p[1]);
  }, { passive: false });
  canvas.addEventListener('keydown', e => {
    const k = e.key, mod = e.ctrlKey || e.metaKey;
    if (mod && k.toLowerCase() === 'z') doUndo();
    else if (mod) return;
    else if (k === 'Delete' || k === 'Backspace') deletePoint();
    else if (k === 'Enter') finishDraw();
    else if (k === 'Escape') { draw = null; sel.v = -1; toolbar(); paint(); }
    else if (k === '+' || k === '=') zoomAt(1.25);
    else if (k === '-') zoomAt(0.8);
    else if (k === 'f') fit();
    else return;
    e.preventDefault(); e.stopPropagation();                 // the notebook must not act on them too
  });
  canvas.addEventListener('pointerleave', () => { mouse = null; if (draw) paint(); });

  // ------------------------------------------------------------------ from Python
  img.onload = () => { fit(); };
  function fromPython() { load(); tree(); toolbar(); img.src = model.get('image') || ''; if (!model.get('image')) fit(); }
  model.on('change:image', () => { img.src = model.get('image') || ''; });
  model.on('change:people', () => { load(); tree(); toolbar(); paint(); });
  model.on('change:photos', () => { photos = model.get('photos') || []; tree(); toolbar(); paint(); });
  model.on('change:current', () => { load(); tree(); toolbar(); });
  model.on('change:status', () => { say(model.get('status')); });
  model.on('msg:custom', msg => {
    if (msg.kind === 'saved') { for (const f in edits) { savedStep[f] = step[f] || 0; delete edits[f]; } tree(); toolbar(); say(msg.text); }
  });
  const ro = new ResizeObserver(() => { const s = view.s; size(); if (s === 1) fit(); else paint(); });
  ro.observe(wrap);
  const guard = ev => { if (dirtyCount()) { ev.preventDefault(); ev.returnValue = ''; } };
  window.addEventListener('beforeunload', guard);
  // for the tutorial's tests: the state, and where a point of the photo is on the screen
  root.__lt = { state: () => ({ view: view, people: people, sel: sel, file: file, draw: draw, unsaved: Object.keys(edits) }),
                screen: q => { const r = canvas.getBoundingClientRect(), p = toScreen(q); return [r.left + p[0], r.top + p[1]]; },
                personAt: q => { const p = toScreen(q), k = hitPerson(p[0], p[1]); return k < 0 ? 0 : people[k].person; } };
  fromPython();
  return () => { ro.disconnect(); window.removeEventListener('beforeunload', guard); };
}

export default { render };"""
        _css = r""".lt{display:flex;flex-wrap:wrap;gap:12px;align-items:flex-start;font:13px/1.4 system-ui,-apple-system,"Segoe UI",sans-serif;color:#1d2330;
  background:#f6f7fa;border:1px solid #d5dae3;border-radius:10px;padding:10px;max-width:100%;box-sizing:border-box}
.lt *{box-sizing:border-box}
.lt-side{flex:0 0 250px;max-height:640px;overflow-y:auto;display:flex;flex-direction:column;gap:6px}
.lt-top{display:flex;gap:6px;flex-wrap:wrap}
.lt-head{font-weight:600;font-size:12px;color:#5a6476;text-transform:uppercase;letter-spacing:.04em;margin-top:6px}
.lt-btn{font:600 12px/1 system-ui,sans-serif;color:#1d2330;background:#fff;border:1px solid #c3cad6;border-radius:6px;
  padding:7px 10px;cursor:pointer;position:relative;overflow:hidden}
.lt-btn:hover{border-color:#5b6cff}
.lt-btn:disabled{opacity:.45;cursor:default}
.lt-btn input{position:absolute;inset:0;opacity:0;cursor:pointer}
.lt-save:not(:disabled){background:#3b4cdb;color:#fff;border-color:#3b4cdb}
.lt-photo{display:flex;gap:8px;align-items:center;padding:6px;border-radius:8px;cursor:pointer;border:1px solid transparent}
.lt-photo:hover{background:#eceff6}
.lt-photo.on{background:#e3e7ff;border-color:#9aa6ff}
.lt-photo img{width:48px;height:32px;object-fit:cover;border-radius:4px;flex:none}
.lt-ptext{flex:1;min-width:0}
.lt-name{font-weight:600;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.lt-sub{font-size:11.5px;color:#5a6476}
.lt-split{font:600 10.5px/1 system-ui,sans-serif;border-radius:10px;padding:4px 7px;border:1px solid;cursor:pointer;background:#fff}
.lt-split.train{color:#1f7a4d;border-color:#7cc7a2}
.lt-split.test{color:#b34700;border-color:#f0a46c}
.lt-people{margin:0 0 4px 18px;display:flex;flex-direction:column;gap:2px}
.lt-person{display:flex;gap:6px;align-items:center;padding:4px 6px;border-radius:6px;cursor:pointer;font-size:12px}
.lt-person:hover{background:#eceff6}
.lt-person.on{background:#dfe4ff;font-weight:600}
.lt-person i{width:10px;height:10px;border-radius:3px;flex:none}
.lt-empty{padding:4px 6px}
.lt-main{flex:1 1 420px;min-width:0;display:flex;flex-direction:column;gap:6px}
.lt-bar{display:flex;gap:6px;flex-wrap:wrap;align-items:center}
.lt-sep{width:1px;height:22px;background:#c3cad6;margin:0 2px}
.lt-wrap{width:100%}
.lt-canvas{display:block;border-radius:8px;outline:none;cursor:crosshair;touch-action:none}
.lt-canvas:focus{box-shadow:0 0 0 2px #9aa6ff}
.lt-status{font:12px/1.4 ui-monospace,Menlo,monospace;color:#3c4654;min-height:17px}"""
        photos = traitlets.List([]).tag(sync=True)      # every photo: file, size, split, people count, a thumbnail
        current = traitlets.Int(0).tag(sync=True)       # the open photo
        image = traitlets.Unicode("").tag(sync=True)    # the open photo itself, as a data URL
        people = traitlets.List([]).tag(sync=True)      # its people, back to front
        status = traitlets.Unicode("").tag(sync=True)

        def __init__(self, dataset):
            super().__init__()
            self.dataset = dataset
            self.on_msg(self._message)
            self._list()
            self._open()

        def _picture(self, i, width):
            """Photo i shrunk to at most `width` pixels across, as a data URL (the outlines stay in photo pixels)."""
            image = Image.open(os.path.join(self.dataset.root, self.dataset.images[i]["file"])).convert("RGB")
            image.thumbnail((width, width * 4))
            buf = io.BytesIO()
            image.save(buf, "JPEG", quality=85)
            return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

        def _list(self):
            d = self.dataset
            self.photos = [dict(file=im["file"], width=im["width"], height=im["height"], split=im["split"],
                                count=len(d.people_of(i)), thumb=self._picture(i, 96)) for i, im in enumerate(d.images)]

        @traitlets.observe("current")
        def _changed(self, change):
            self._open()

        def _open(self):
            if not len(self.dataset):
                self.people, self.image = [], ""
                return
            i = min(self.current, len(self.dataset) - 1)
            self.people = [dict(person=p["person"], polygons=p["polygons"], holes=p.get("holes", []))
                           for p in self.dataset.people_of(i)]
            self.image = self._picture(i, 1600)

        def _message(self, widget, msg, buffers):
            try:
                self._handle(msg)
            except Exception as e:                      # shown in the tool: an error here would otherwise be silent
                self.status = f"Error: {type(e).__name__}: {e}"

        def _handle(self, msg):
            d = self.dataset
            if msg["kind"] == "save":
                for name, people in msg["edits"].items():
                    d.set_people(d.index(name), people)
                d.save()
                self._list()
                self._open()
                self.send({"kind": "saved", "text": f"Saved {len(d)} photos, {len(d.people)} people -> {d.path}"})
            elif msg["kind"] == "upload":
                i = d.add_photo(msg["name"], base64.b64decode(msg["data"]))
                d.save()
                self._list()
                if self.current == i:                   # the first photo of an empty dataset: no change to observe
                    self._open()
                self.current = i
                self.status = f"Added {d.images[i]['file']}: trace its people, then Save."
            elif msg["kind"] == "split":
                d.images[d.index(msg["file"])]["split"] = msg["split"]
                d.save()
                self._list()
            elif msg["kind"] == "remove":
                d.remove_photo(d.index(msg["file"]))
                d.save()
                self._list()
                self.current = max(0, min(self.current, len(d) - 1))
                self._open()
                self.status = f"Took {msg['file']} out of the dataset (the file is still in the folder)."

    if IN_NOTEBOOK:
        tool = LabelTool(dataset)
        display(tool)

## 4. Data: the dataset, many pictures
Two photos are far too few, so every training picture is made on the fly from one of them: zoomed in or out (0.5x to
1.5x), moved, sometimes mirrored, with brightness, contrast and colour strength changed, and half the time with one
person cut out of a photo and pasted somewhere else (copy-paste). Where a zoomed-out photo does not fill the picture,
the gap is padding: it gets the label IGNORE, so it costs no loss and is never scored.

`AugmentedPictures` holds the photos training may see — any number, of any shape: a photo that is not 3:2 is
letterboxed — and makes whole batches at once on the GPU: `affine_grid` and `grid_sample` move, zoom and mirror 32
pictures in one call. It is a PyTorch `IterableDataset`; the `DataLoader` around it hands the training loop one epoch
of batches.

In [ ]:
# ---- which photos (and which part of them) training may see, and what the tests score
def splits(dataset, split):
    """Two lists: the training parts, (photo, hide everything right of x — or None), and the tests, (photo, score
    only from x on)."""
    names = [n for i, n in enumerate(dataset.names()) if dataset.people_of(i)]
    if len(names) < len(dataset):                    # a photo just added has no people yet: it would teach "nobody"
        print(f"Left out {len(dataset) - len(names)} photo(s) with no people traced yet")
    if split == "both":                              # the final model: every photo; its scores are on what it learned
        return [(n, None) for n in names], [(n, 0) for n in names]
    if split == "photo1":                            # a new photo: other people, other wall, other floor
        return [(names[0], None)], [(n, 0) for n in names[1:]]
    if split == "right_out":                         # new people: the two at the right of each photo are hidden
        return [(n, CUT_X[n]) for n in names if n in CUT_X], [(n, CUT_X[n]) for n in names if n in CUT_X]
    if split == "labels":                            # as marked in the labelling tool
        marked = lambda s: [im["file"] for im in dataset.images if im["file"] in names and im["split"] == s]
        return [(n, None) for n in marked("train")], [(n, 0) for n in marked("test")]
    raise ValueError(split)


train_parts, tests = splits(dataset, SPLIT)


def training_photo(dataset, name, keep_left_of=None):
    """The photo at BASE size, its mask, and which pixels training may use (all, or those left of the cut). A photo
    that is not 3:2 is shrunk to fit and centred; the rest of the canvas is not usable."""
    i = dataset.index(name)
    w, h = dataset.images[i]["width"], dataset.images[i]["height"]
    if abs(w / h - BASE_W / BASE_H) < 0.01:          # 3:2 within 1%, as both photos are: it fills the canvas
        nw, nh = BASE_W, BASE_H
    else:
        s = min(BASE_W / w, BASE_H / h)
        nw, nh = round(w * s), round(h * s)
    image, mask = dataset.photo(i, (nw, nh)), dataset.mask(i, (nw, nh))
    usable = torch.ones(1, nh, nw)
    if keep_left_of is not None:                     # right of the cut: blanked out, never seen in training
        usable[:, :, round(keep_left_of * nw / w):] = 0
        image, mask = image * usable, mask * usable[0].long()
    if (nw, nh) != (BASE_W, BASE_H):                 # letterbox: centred on an empty canvas
        top, left = (BASE_H - nh) // 2, (BASE_W - nw) // 2
        rows, cols = slice(top, top + nh), slice(left, left + nw)
        canvas_photo = torch.zeros(3, BASE_H, BASE_W)
        canvas_mask = torch.zeros(BASE_H, BASE_W, dtype=torch.long)
        canvas_usable = torch.zeros(1, BASE_H, BASE_W)          # the bars around the photo stay 0: not usable
        canvas_photo[:, rows, cols] = image
        canvas_mask[rows, cols] = mask
        canvas_usable[:, rows, cols] = usable
        image, mask, usable = canvas_photo, canvas_mask, canvas_usable
    return image, mask, usable


def uniform(low, high, n):
    """n random numbers between low and high, on the device."""
    return torch.empty(n, device=device).uniform_(low, high)


def placement(scale, flip, shift_x, shift_y):
    """One 2 x 3 matrix per picture for affine_grid: show the source `scale` times larger, mirrored when flip = -1,
    moved by (shift_x, shift_y). Coordinates run from -1 to 1 across the picture."""
    theta = torch.zeros(len(scale), 2, 3, device=device)
    theta[:, 0, 0], theta[:, 0, 2] = flip / scale, -shift_x * flip / scale
    theta[:, 1, 1], theta[:, 1, 2] = 1 / scale, -shift_y / scale
    return theta


def warp(source, theta, mode):
    """source (n, channels, H, W) zoomed, mirrored and moved by theta, at the model's picture size. mode "bilinear"
    blends neighbouring pixels (photos); "nearest" copies one (labels, which must never be blended)."""
    grid = F.affine_grid(theta, (len(theta), source.size(1), IMG_H, IMG_W), align_corners=False)
    return F.grid_sample(source, grid, mode=mode, padding_mode="zeros", align_corners=False)


class AugmentedPictures(IterableDataset):
    """The training pictures, made on the fly from the photos training may see. Iterating gives one epoch:
    `batches` batches of (pictures (n, 3, IMG_H, IMG_W), labels (n, IMG_H, IMG_W): 0, 1 or IGNORE)."""

    def __init__(self, dataset, parts, batch_size, batches):
        made = [training_photo(dataset, name, cut) for name, cut in parts]
        self.images = torch.stack([m[0] for m in made]).to(device)                  # (P, 3, BASE_H, BASE_W)
        self.masks = torch.stack([m[1] for m in made])[:, None].float().to(device)  # (P, 1, BASE_H, BASE_W): person numbers
        self.usable = torch.stack([m[2] for m in made]).to(device)                  # (P, 1, BASE_H, BASE_W): 1 = may be used
        # every colour channel's mean and spread over the training pixels, to normalise with
        pixels = self.images.permute(1, 0, 2, 3).flatten(1)[:, self.usable.flatten() > 0]
        self.mean, self.std = pixels.mean(1), pixels.std(1)
        # every person training may see, for copy-paste: (photo, person number, centre x, centre y), centre in [-1, 1]
        self.cutouts = []
        for p in range(len(made)):
            for k in self.masks[p].unique().int().tolist():
                if k > 0:
                    ys, xs = torch.where(self.masks[p, 0] == k)
                    self.cutouts.append((p, k, (xs.float().mean().item() + 0.5) / BASE_W * 2 - 1,
                                         (ys.float().mean().item() + 0.5) / BASE_H * 2 - 1))
        # the same four columns as tensors on the device, which _paste picks from
        photo_of, person_of, x_of, y_of = zip(*self.cutouts)
        self.cut_photo = torch.tensor(photo_of, device=device)
        self.cut_person = torch.tensor(person_of, device=device).float()
        self.cut_x, self.cut_y = torch.tensor(x_of, device=device), torch.tensor(y_of, device=device)
        self.batch_size, self.batches = batch_size, batches

    def batch(self, n):
        """n new training pictures, normalised, and their labels: 0 background, 1 person, IGNORE padding."""
        x, labels, usable = self._place(n)
        x, labels, usable = self._paste(x, labels, usable, n)
        x = self._recolour(x, n)
        return self._finish(x, labels, usable)

    def _place(self, n):
        """Step 1: a photo for each picture, zoomed (SCALE_RANGE), mirrored half the time and moved; the photo, its
        labels and its usable map are warped together, so they stay aligned."""
        which = torch.randint(len(self.images), (n,), device=device)
        scale = torch.exp(uniform(math.log(SCALE_RANGE[0]), math.log(SCALE_RANGE[1]), n))
        flip = torch.where(torch.rand(n, device=device) < 0.5, -1.0, 1.0)
        room = (scale - 1).abs()                     # how far it can move and still fill (or stay inside) the picture
        theta = placement(scale, flip, uniform(-1, 1, n) * room, uniform(-1, 1, n) * room)
        x = warp(self.images[which], theta, "bilinear")
        labels = warp(self.masks[which], theta, "nearest")
        usable = warp(self.usable[which], theta, "nearest")
        return x, labels, usable

    def _paste(self, x, labels, usable, n):
        """Step 2, copy-paste: in a share PASTE_PROB of the pictures, one person — zoomed and mirrored at random —
        is dropped somewhere new."""
        pick = torch.randint(len(self.cutouts), (n,))
        src, who = self.cut_photo[pick], self.cut_person[pick].view(-1, 1, 1, 1)
        cx, cy = self.cut_x[pick], self.cut_y[pick]
        p_scale = torch.exp(uniform(math.log(PASTE_SCALE[0]), math.log(PASTE_SCALE[1]), n))
        p_flip = torch.where(torch.rand(n, device=device) < 0.5, -1.0, 1.0)
        to_x, to_y = uniform(*PASTE_AT_X, n), uniform(*PASTE_AT_Y, n)      # where the person's centre lands
        p_theta = placement(p_scale, p_flip, to_x - p_scale * cx * p_flip, to_y - p_scale * cy)
        cut = ((self.masks[src] == who) & (self.usable[src] > 0)).float()
        pasted = (warp(cut, p_theta, "bilinear") > 0.5) & (torch.rand(n, 1, 1, 1, device=device) < PASTE_PROB)
        x = torch.where(pasted, warp(self.images[src], p_theta, "bilinear"), x)
        labels = torch.where(pasted, torch.ones_like(labels), labels)
        usable = torch.where(pasted, torch.ones_like(usable), usable)
        return x, labels, usable

    def _recolour(self, x, n):
        """Step 3: brightness, contrast and colour strength, a little different in every picture."""
        brightness, contrast, saturation = (uniform(lo, hi, n).view(-1, 1, 1, 1) for lo, hi in
                                            (BRIGHTNESS, CONTRAST, SATURATION))
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        x = ((x - mean) * contrast + mean) * brightness
        grey = x.mean(1, keepdim=True)
        return (grey + (x - grey) * saturation).clamp(0, 1)

    def _finish(self, x, labels, usable):
        """Step 4: padding gets the average colour (0 once normalised) and the label IGNORE, every person number
        becomes 1, and the pictures are normalised."""
        usable = usable > 0.5
        x = torch.where(usable, x, self.mean.view(1, -1, 1, 1))
        y = torch.where(usable[:, 0], (labels[:, 0] > 0).long(), torch.full_like(labels[:, 0], IGNORE).long())
        return (x - self.mean.view(1, -1, 1, 1)) / self.std.view(1, -1, 1, 1), y

    def __iter__(self):
        for _ in range(self.batches):
            yield self.batch(self.batch_size)


pictures = AugmentedPictures(dataset, train_parts, BATCH_SIZE, SAMPLES_PER_EPOCH // BATCH_SIZE)
MEAN, STD = pictures.mean, pictures.std


def normalize(x):
    """One picture (3, H, W) with the training pictures' colour statistics: what the model is always given."""
    return (x - MEAN.view(-1, 1, 1).to(x.device)) / STD.view(-1, 1, 1).to(x.device)


# batch_size=None: the pictures come in batches already; its own generator leaves the training's random numbers alone
train_loader = DataLoader(pictures, batch_size=None, generator=torch.Generator())

x, y = pictures.batch(8)
print(f"Training from {len(pictures.images)} photo(s), {len(pictures.cutouts)} people to paste; one batch: "
      f"{tuple(x.shape)} pictures, {(y == 1).float().mean():.0%} person, {(y == 0).float().mean():.0%} background, "
      f"{(y == IGNORE).float().mean():.0%} padding")

fig, axes = plt.subplots(2, 4, figsize=(12, 4.2))
for ax, picture, label in zip(axes.flat, x.cpu(), y.cpu()):
    rgb = (picture * STD.cpu().view(-1, 1, 1) + MEAN.cpu().view(-1, 1, 1)).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(rgb)
    ax.imshow(np.ma.masked_where(label.numpy() != 1, label.numpy()), cmap="autumn", alpha=0.45)
    ax.axis("off")
plt.suptitle("Eight training pictures made from the photos (people tinted)")
plt.tight_layout()
plt.show()

## 5. Model
**Encoder:** a convolutional stem (three 3 x 3 convolutions, each halving the picture: 192 x 288 -> 96 x 144 -> 48 x 72
-> 24 x 36), then pre-norm encoder layers (attention, then a small GELU network) and a final LayerNorm. There is no
table of positions: inside every attention the queries and keys are turned by their row and column (2-D RoPE), so a
score depends on where two patches are relative to each other, not on where they are in the picture.
**Decoder:** the stem's 48 x 72 map gives every 4 x 4-pixel cell a query; decoder layers let each cell attend to all
864 encoder patches (cross-attention), then think it over; a linear layer gives every cell a score for background and
for person, and the 48 x 72 scores are stretched to 192 x 288 (bilinear) — four times less stretching than v1's.

In [ ]:
# ============================================================
# [NEW v1.5] Pixels -> patch vectors: a convolutional stem
# ============================================================

class ConvStem(nn.Module):
    """[NEW v1.5] Replaces v1's PatchEmbedding, which cut the picture into hard 8 x 8 squares. Three 3 x 3
    convolutions, each with stride 2, halve the picture three times: 192 x 288 -> 96 x 144 -> 48 x 72 -> 24 x 36.
    Each output looks at a 3 x 3 window of the map before it and neighbouring windows overlap: a patch reads its own
    8 x 8 pixels and 7 more above it and to its left. Each convolution is followed by BatchNorm (keeps its outputs steady in
    size) and GELU; a final 1 x 1 convolution maps to d_model. The 48 x 72 map in the middle — one vector per
    4 x 4 pixels — is returned too: the decoder starts from it."""
    def __init__(self, in_ch: int, channels: tuple, d_model: int):
        super().__init__()
        c1, c2 = channels

        def block(c_in, c_out):                     # 3 x 3, stride 2: half the height and width
            return nn.Sequential(nn.Conv2d(c_in, c_out, kernel_size=3, stride=2, padding=1, bias=False),
                                 nn.BatchNorm2d(c_out), nn.GELU())
        self.conv1, self.conv2, self.conv3 = block(in_ch, c1), block(c1, c2), block(c2, d_model)
        self.proj = nn.Conv2d(d_model, d_model, kernel_size=1)

    def forward(self, x):                           # (B, 3, H, W)
        fine = self.conv2(self.conv1(x))            # (B, c2, H/4, W/4): one vector per 4 x 4 pixels
        x = self.proj(self.conv3(fine))             # (B, d_model, H/8, W/8): one per 8 x 8 patch, as in v1
        return x.flatten(2).transpose(1, 2), fine   # (B, N, d_model), one row per patch; and the finer map


# ============================================================
# [NEW v1.5] Positions as rotations, in two directions (2-D RoPE)
# ============================================================

def rope_tables_2d(head_dim: int, rows, cols, base: float = 100.0):
    """[NEW v1.5] The RoPE of text v2.5 in two dimensions. A head's d_k numbers are read as d_k / 2 pairs: the first
    half of the pairs turn with the row, the second half with the column, and pair i of a half turns by
    position * base ** (-i / half) — fast to slow. rows, cols: (N,) positions in patches (not always whole numbers).
    Returns cos and sin of every angle, (N, d_k // 2) each."""
    half = head_dim // 4                            # pairs per direction
    speed = base ** (-torch.arange(half, dtype=torch.float) / half)
    angle = torch.cat([torch.outer(rows, speed), torch.outer(cols, speed)], dim=1)
    return angle.cos(), angle.sin()


def apply_rope(x, cos, sin):
    """[NEW v1.5] Turn every pair (x[..., 2i], x[..., 2i+1]) of x (B, H, N, d_k) by its angle — text v2.5's function.
    Turning a query and a key by their own positions makes q . k depend on where the two are relative to each other."""
    x1, x2 = x[..., 0::2], x[..., 1::2]
    turned = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return turned.flatten(-2)


def grid_positions(n_rows, n_cols, size, patch):
    """[NEW v1.5] The centres of an n_rows x n_cols grid of size x size squares, in patches: the patches themselves
    sit at 0, 1, 2, …; a 4 x 4 cell r covers pixels 4r … 4r + 3, so its centre is at (r - 0.5) / 2. Row by row."""
    r = (torch.arange(n_rows, dtype=torch.float) * size + (size - patch) / 2) / patch
    c = (torch.arange(n_cols, dtype=torch.float) * size + (size - patch) / 2) / patch
    rows, cols = torch.meshgrid(r, c, indexing="ij")
    return rows.flatten(), cols.flatten()


# ============================================================
# Multi-head attention  (no mask: every query may look at every key)
#   [NEW v1.5] the same class serves self-attention (encoder) and cross-attention (decoder)
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads, self.d_k = num_heads, d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)

    def _split(self, x):                            # (B, N, D) -> (B, H, N, d_k)
        return x.view(x.size(0), -1, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x, rope, memory=None, memory_rope=None, return_attn=False):
        """[NEW v1.5] Queries come from x. Keys and values come from x too (self-attention: the encoder's patches) or
        from memory (cross-attention: x = the decoder's cells, memory = the encoder's patches). rope = (cos, sin) for
        the positions of x, memory_rope for those of memory."""
        if memory is None:
            memory, memory_rope = x, rope
        Q = apply_rope(self._split(self.W_q(x)), *rope)                 # [NEW v1.5] turned by its position
        K = apply_rope(self._split(self.W_k(memory)), *memory_rope)     # [NEW v1.5] turned by its position
        V = self._split(self.W_v(memory))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)    # (B, H, N_q, N_k)
        weights = torch.softmax(scores, dim=-1)
        context = self.attn_dropout(weights) @ V                   # (B, H, N_q, d_k)
        context = context.transpose(1, 2).contiguous().view(x.size(0), -1, self.num_heads * self.d_k)
        out = self.W_o(context)
        return (out, weights) if return_attn else out


# ============================================================
# One encoder layer (pre-norm)
# ============================================================

class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, rope, return_attn=False):
        a = self.attn(self.norm1(x), rope, return_attn=return_attn)    # [NEW v1.5] rope: the patches' rotations
        a, w = a if return_attn else (a, None)
        x = x + self.dropout(a)
        x = x + self.dropout(self.mlp(self.norm2(x)))
        return x, w


# ============================================================
# [NEW v1.5] One decoder layer (pre-norm): each cell asks the patches what it is
# ============================================================

class DecoderLayer(nn.Module):
    """[NEW v1.5] Cross-attention — the cells' queries against the encoder's keys and values, as the text decoder asks
    its encoder — then a small GELU network, each with a residual. The text decoder also lets its tokens attend to
    each other first; here that would be 3,456 x 3,456 scores per head for every picture, so the cells meet only
    through the patches they read."""
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, rope, memory_rope, return_attn=False):
        a = self.cross_attn(self.norm1(x), rope, memory, memory_rope, return_attn)
        a, w = a if return_attn else (a, None)
        x = x + self.dropout(a)
        x = x + self.dropout(self.mlp(self.norm2(x)))
        return x, w


# ============================================================
# The whole model  [NEW v1.5] an encoder and a decoder
# ============================================================

class SegViT(nn.Module):
    def __init__(self, img_h=192, img_w=288, patch=8, in_ch=3, num_classes=2, d_model=128,
                 num_heads=4, num_layers=4, d_ff=512, dropout=0.1,
                 stem_ch=(32, 64), cell=4, dec_layers=2, rope_base=100.0):     # [NEW v1.5]
        super().__init__()
        assert patch == 8 and cell == 4, "the stem halves the picture 3 times (8 x 8 patches) and keeps the 4 x 4 map"
        self.img_h, self.img_w = img_h, img_w
        self.grid_h, self.grid_w = img_h // patch, img_w // patch    # 24 x 36 patches
        self.cell_h, self.cell_w = img_h // cell, img_w // cell      # [NEW v1.5] 48 x 72 cells of 4 x 4 pixels
        self.stem = ConvStem(in_ch, stem_ch, d_model)               # [NEW v1.5] replaces PatchEmbedding
        # [NEW v1.5] no pos_embed: every attention turns its queries and keys by their row and column instead
        d_k = d_model // num_heads
        for name, (rows, cols) in (("patch", grid_positions(self.grid_h, self.grid_w, patch, patch)),
                                   ("cell", grid_positions(self.cell_h, self.cell_w, cell, patch))):
            cos, sin = rope_tables_2d(d_k, rows, cols, rope_base)
            self.register_buffer(name + "_cos", cos, persistent=False)          # [NEW v1.5]
            self.register_buffer(name + "_sin", sin, persistent=False)          # [NEW v1.5]
        self.dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.cell_in = nn.Linear(stem_ch[1], d_model)               # [NEW v1.5] a cell's query, from its 4 x 4 pixels
        self.dec_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout)       # [NEW v1.5]
                                         for _ in range(dec_layers)])
        self.dec_norm = nn.LayerNorm(d_model)                       # [NEW v1.5]
        self.head = nn.Linear(d_model, num_classes)                 # background and person scores for every cell

    def forward(self, images, return_attn=False):
        patches, fine = self.stem(images)                            # [NEW v1.5] (B, N, d_model), (B, c2, 48, 72)
        x = self.dropout(patches)                                    # no positions added: they are rotations now
        rope = (self.patch_cos, self.patch_sin)
        attns = []
        for layer in self.layers:
            x, w = layer(x, rope, return_attn)
            attns.append(w)
        memory = self.norm(x)                                        # [NEW v1.5] what the encoder read, per patch
        q = self.dropout(self.cell_in(fine.flatten(2).transpose(1, 2)))   # [NEW v1.5] (B, 3456, d_model)
        cross = []
        for layer in self.dec_layers:                                # [NEW v1.5] each cell asks the patches
            q, w = layer(q, memory, (self.cell_cos, self.cell_sin), rope, return_attn)
            cross.append(w)
        logits = self.head(self.dec_norm(q))                         # (B, cells, classes)
        B, N, C = logits.shape
        logits = logits.transpose(1, 2).reshape(B, C, self.cell_h, self.cell_w)    # [NEW v1.5] (B, classes, 48, 72)
        logits = F.interpolate(logits, size=(self.img_h, self.img_w), mode="bilinear", align_corners=False)
        return (logits, attns, cross) if return_attn else logits    # (B, classes, H, W)


# ============================================================
# Measuring a segmentation
# ============================================================

def confusion(pred, target, n=NUM_CLASSES):
    """conf[t, p] = how many pixels of true class t were given class p (IGNORE pixels left out)."""
    keep = target != IGNORE
    return torch.bincount(target[keep] * n + pred[keep], minlength=n * n).reshape(n, n)


def scores(conf):
    """Pixel accuracy, IoU per class (overlap / union) and their mean, mIoU."""
    tp = conf.diag().float()
    union = (conf.sum(0) + conf.sum(1) - conf.diag()).float()
    iou = tp / union.clamp(min=1)
    return (tp.sum() / conf.sum()).item(), iou, iou.mean().item()

## 6. Model, optimizer and loss
AdamW with a short warmup and a cosine decay. The loss is cross-entropy for every pixel: the two scores of a pixel
against its true label, averaged over the pixels. `ignore_index=IGNORE` leaves the padding out of the average.

In [ ]:
model_config = dict(img_h=IMG_H, img_w=IMG_W, patch=PATCH, in_ch=3, num_classes=NUM_CLASSES, d_model=D_MODEL,
                    num_heads=NUM_HEADS, num_layers=NUM_LAYERS, d_ff=D_FF, dropout=DROPOUT,
                    stem_ch=STEM_CH, cell=CELL, dec_layers=DEC_LAYERS, rope_base=ROPE_BASE)    # [NEW v1.5]
model = SegViT(**model_config).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = SAMPLES_PER_EPOCH // BATCH_SIZE
total_steps = EPOCHS * steps_per_epoch
warmup_steps = max(1, total_steps // 20)

def lr_lambda(step):                                # warmup, then cosine decay
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE)

## 7. Training
Every epoch makes SAMPLES_PER_EPOCH new pictures. After each epoch the test photos are scored, only to draw the
curve: with two photos there is nothing left over to pick a "best" epoch with, so the model kept is the last one.

In [ ]:
def segment(image):
    """Person scores for a photo of any size, (3, h, w) in [0, 1]: letterboxed into IMG_W x IMG_H (padding = average
    colour), run through the model, and the scores stretched back to h x w. Returns logits (2, h, w)."""
    h, w = image.shape[-2:]
    s = min(IMG_W / w, IMG_H / h)
    nw, nh = round(w * s), round(h * s)
    small = F.interpolate(image[None], size=(nh, nw), mode="bilinear", antialias=True, align_corners=False)[0]
    canvas = MEAN.view(-1, 1, 1).cpu().repeat(1, IMG_H, IMG_W)
    top, left = (IMG_H - nh) // 2, (IMG_W - nw) // 2
    canvas[:, top:top + nh, left:left + nw] = small
    with torch.no_grad():
        logits = model(normalize(canvas)[None].to(device))[0, :, top:top + nh, left:left + nw]
    return F.interpolate(logits[None], size=(h, w), mode="bilinear", align_corners=False)[0].cpu()


def test_scores():
    """Person IoU and pixel accuracy on each test photo (the part right of x0), at the photo's full size."""
    out = {}
    for name, x0 in tests:
        pred = segment(photos[name]).argmax(0)[:, x0:]
        true = (masks[name][:, x0:] > 0).long()
        acc, iou, _ = scores(confusion(pred, true))
        out[name] = (iou[1].item(), acc)
    return out


start = time.time()
for epoch in range(EPOCHS):
    model.train()
    total_loss, conf = 0.0, torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    for images, labels in train_loader:                              # one epoch: steps_per_epoch new batches
        logits = model(images)                                       # (B, classes, H, W)
        loss = criterion(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        conf += confusion(logits.argmax(1).cpu(), labels.cpu())
    model.eval()
    _, train_iou, _ = scores(conf)
    tested = test_scores()
    print(f"Epoch {epoch:3d} | train loss {total_loss / steps_per_epoch:.4f} person IoU {train_iou[1]:.1%} | test "
          + " ".join(f"{name} IoU {iou:.1%}" for name, (iou, _) in tested.items()) + f" | {time.time() - start:.0f}s")

torch.save({"model": model.state_dict(), "config": model_config, "split": SPLIT, "epochs": EPOCHS,
            "mean": MEAN.cpu(), "std": STD.cpu()}, CKPT_PATH)

## 8. Evaluation
The person IoU (person pixels found and labelled, over all pixels that are a person in the truth or the prediction),
pixel accuracy, and how much of each person was found. What the numbers mean depends on SPLIT: with "both" the model
has seen these photos, so they show how well it learned them, not how it does on a new photo; "photo1" and
"right_out" score photos or people it never saw; "labels" scores the photos you marked *test* in the labelling tool.

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device)
model = SegViT(**ckpt["config"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

seen = {"both": "seen in training", "photo1": "a photo it never saw", "right_out": "people it never saw",
        "labels": "photos marked test"}[SPLIT]
for name, x0 in tests:
    pred = segment(photos[name]).argmax(0)[:, x0:]
    instance = masks[name][:, x0:]
    pixel_acc, iou, miou = scores(confusion(pred, (instance > 0).long()))
    where = f"right of x = {x0}" if x0 else "whole photo"
    print(f"{name} ({where}; {seen}): person IoU {iou[1]:.1%} | background IoU {iou[0]:.1%} | "
          f"pixel accuracy {pixel_acc:.1%}")
    for k in instance.unique().tolist():
        if k > 0:
            inside = instance == k
            print(f"  person {k}: {(pred[inside] == 1).float().mean():.1%} of their pixels found")

## 9. Predictions: paint the people
`segment_photo` takes any photo, finds the person pixels and paints them in the colour you choose. Try your own:
`segment_photo("my_photo.jpg", color="#00c853")`. It learned from two studio photos, so expect it to struggle on
scenes that look nothing like them.

In [ ]:
def segment_photo(path, color="#ff2d55", alpha=0.55, show=True):
    """Paints the people in a photo. Returns the person mask (h, w) as a bool tensor."""
    image = image_to_tensor(Image.open(path).convert("RGB"))
    person = segment(image).argmax(0) == 1
    paint = torch.tensor(to_rgb(color)).view(3, 1, 1)               # "#ff2d55" -> (1.0, 0.18, 0.33); names work too
    painted = torch.where(person, (1 - alpha) * image + alpha * paint, image)
    if show:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        axes[0].imshow(image.permute(1, 2, 0))
        axes[0].set_title(os.path.basename(path))
        axes[1].imshow(painted.permute(1, 2, 0))
        axes[1].set_title(f"people: {person.float().mean():.1%} of the pixels")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
    return person


for name in PHOTOS:
    segment_photo(os.path.join(DATA_DIR, name))

## 10. Attention map
Where one patch looks, in the encoder's last layer, and — new in v1.5 — which patches one 4 x 4-pixel cell reads, in the
decoder's last layer; both averaged over the heads, at the middle of a person in photo 1.

In [ ]:
def plot_attention(name=PHOTOS[0], person=4):
    image = load_photo(name, (IMG_W, IMG_H))
    with torch.no_grad():
        _, attns, cross = model(normalize(image)[None].to(device), return_attn=True)   # [NEW v1.5] cross: the decoder's
    small = load_mask(name, (IMG_W, IMG_H))
    ys, xs = torch.where(small == person)
    row, col = int(ys.float().mean()) // PATCH, int(xs.float().mean()) // PATCH   # the patch at the person's centre
    patch = row * model.grid_w + col
    weights = attns[-1][0].mean(0)[patch].reshape(model.grid_h, model.grid_w).cpu()
    crow, ccol = int(ys.float().mean()) // CELL, int(xs.float().mean()) // CELL  # [NEW v1.5] the cell at that place
    cell = crow * model.cell_w + ccol
    reads = cross[-1][0].mean(0)[cell].reshape(model.grid_h, model.grid_w).cpu()   # [NEW v1.5] over the 864 patches
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))                         # [NEW v1.5] a third picture
    axes[0].imshow(image.permute(1, 2, 0))
    axes[0].add_patch(plt.Rectangle((col * PATCH - 0.5, row * PATCH - 0.5), PATCH, PATCH, fill=False, color="white", lw=2))
    axes[0].set_title(f"{name}, patch {patch} (person {person})")
    axes[1].imshow(image.permute(1, 2, 0))
    axes[1].imshow(weights, cmap="viridis", alpha=0.75, extent=(-0.5, IMG_W - 0.5, IMG_H - 0.5, -0.5))
    axes[1].set_title("where the patch looks (encoder, last layer)")
    axes[2].imshow(image.permute(1, 2, 0))                                    # [NEW v1.5]
    axes[2].imshow(reads, cmap="viridis", alpha=0.75, extent=(-0.5, IMG_W - 0.5, IMG_H - 0.5, -0.5))
    axes[2].add_patch(plt.Rectangle((ccol * CELL - 0.5, crow * CELL - 0.5), CELL, CELL, fill=False, color="white", lw=2))
    axes[2].set_title(f"what cell {cell} reads (decoder, last layer)")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_attention()